In [0]:
dbutils.widgets.text("catalog", "bank_risk_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.mv_credit_risk
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Riesgo de crédito por mes. Los saldos son fotos de cierre de mes, por eso siempre se agrupan o filtran por mes."
source: {catalog}.gold.fact_loan_month
joins:
  - name: loan
    source: {catalog}.gold.dim_loan
    'on': source.loan_id = loan.loan_id
  - name: customer
    source: {catalog}.gold.dim_customer
    'on': source.customer_id = customer.customer_id
fields:
  - name: mes
    expr: CAST(DATE_TRUNC('MONTH', period_end) AS DATE)
    display_name: Mes
    synonyms: [periodo, mes de corte, fecha de corte]
  - name: segmento
    expr: customer.segment
    display_name: Segmento
    synonyms: [segmento de cliente, tipo de cliente]
  - name: estado
    expr: customer.state
    display_name: Estado
    synonyms: [entidad, entidad federativa]
  - name: rango_ingreso
    expr: customer.income_band
    display_name: Rango de ingreso
  - name: producto
    expr: loan.product
    display_name: Producto
    synonyms: [tipo de crédito, producto de crédito]
  - name: calificacion
    expr: loan.risk_grade
    display_name: Calificación de riesgo
    synonyms: [rating, grado de riesgo]
  - name: tramo_atraso
    expr: dpd_bucket
    display_name: Tramo de días de atraso
    synonyms: [bucket de atraso, días de mora]
  - name: estatus
    expr: status
    display_name: Estatus del crédito
measures:
  - name: cartera_total
    expr: SUM(balance) FILTER (WHERE status IN ('vigente', 'vencido'))
    display_name: Cartera total
    comment: Saldo de créditos vigentes y vencidos al cierre del mes
    synonyms: [saldo total, cartera, portafolio]
  - name: cartera_vencida
    expr: SUM(balance) FILTER (WHERE status = 'vencido')
    display_name: Cartera vencida
    comment: Saldo de créditos con 90 días o más de atraso
    synonyms: [saldo vencido, cartera en mora]
  - name: imor
    expr: SUM(balance) FILTER (WHERE status = 'vencido') / NULLIF(SUM(balance) FILTER (WHERE status IN ('vigente', 'vencido')), 0)
    display_name: IMOR
    comment: Índice de morosidad, cartera vencida entre cartera total
    synonyms: [índice de morosidad, morosidad, NPL ratio, tasa de morosidad]
  - name: creditos_activos
    expr: COUNT(DISTINCT loan.loan_id) FILTER (WHERE status IN ('vigente', 'vencido'))
    display_name: Créditos activos
  - name: creditos_vencidos
    expr: COUNT(DISTINCT loan.loan_id) FILTER (WHERE status = 'vencido')
    display_name: Créditos vencidos
  - name: monto_castigado
    expr: SUM(balance) FILTER (WHERE status = 'castigado')
    display_name: Monto castigado
    comment: Saldo enviado a castigo en el mes, 180 días o más de atraso
    synonyms: [castigos, quebrantos, write-off]
  - name: tasa_cobranza
    expr: SUM(amount_paid) / NULLIF(SUM(amount_due), 0)
    display_name: Tasa de cobranza
    comment: Pago recibido entre pago exigible del mes
    synonyms: [cobranza, efectividad de cobranza]
$$
""")

In [0]:
display(spark.sql(f"""
SELECT mes, segmento, MEASURE(cartera_total) AS cartera_total, MEASURE(imor) AS imor
FROM {catalog}.gold.mv_credit_risk
WHERE segmento = 'masivo'
GROUP BY mes, segmento
ORDER BY mes
"""))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.mv_fraud
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Transferencias SPEI enviadas y fraude confirmado por mes, canal, banco destino y cliente."
source: {catalog}.gold.fact_transfer
joins:
  - name: customer
    source: {catalog}.gold.dim_customer
    'on': source.customer_id = customer.customer_id
  - name: bank
    source: {catalog}.gold.dim_bank
    'on': source.dest_bank = bank.bank_code
fields:
  - name: mes
    expr: CAST(DATE_TRUNC('MONTH', transfer_date) AS DATE)
    display_name: Mes
    synonyms: [periodo]
  - name: fecha
    expr: transfer_date
    display_name: Fecha
    synonyms: [día, fecha de operación]
  - name: canal
    expr: channel
    display_name: Canal
    synonyms: [medio, app, web, sucursal]
  - name: hora
    expr: transfer_hour
    display_name: Hora del día
    synonyms: [horario]
  - name: banco_destino
    expr: bank.bank_name
    display_name: Banco destino
    synonyms: [banco receptor, banco beneficiario]
  - name: segmento
    expr: customer.segment
    display_name: Segmento
    synonyms: [segmento de cliente, tipo de cliente]
  - name: estado
    expr: customer.state
    display_name: Estado
    synonyms: [entidad, entidad federativa]
  - name: beneficiario_nuevo
    expr: new_beneficiary
    display_name: Beneficiario nuevo
  - name: dispositivo_nuevo
    expr: new_device
    display_name: Dispositivo nuevo
measures:
  - name: transferencias
    expr: COUNT(1)
    display_name: Transferencias
    synonyms: [operaciones, número de transferencias, SPEI]
  - name: monto_transferido
    expr: SUM(amount)
    display_name: Monto transferido
    synonyms: [importe, volumen]
  - name: transferencias_con_alerta
    expr: COUNT(1) FILTER (WHERE has_alert)
    display_name: Transferencias con alerta
  - name: tasa_alerta
    expr: COUNT(1) FILTER (WHERE has_alert) / COUNT(1)
    display_name: Tasa de alerta
    comment: Transferencias con alerta entre total de transferencias
  - name: fraudes_confirmados
    expr: COUNT(1) FILTER (WHERE is_confirmed_fraud)
    display_name: Fraudes confirmados
    synonyms: [fraudes, casos de fraude]
  - name: monto_fraude
    expr: SUM(amount) FILTER (WHERE is_confirmed_fraud)
    display_name: Monto de fraude
    synonyms: [pérdida por fraude, monto defraudado]
  - name: tasa_fraude
    expr: COUNT(1) FILTER (WHERE is_confirmed_fraud) / COUNT(1)
    display_name: Tasa de fraude
    comment: Fraudes confirmados entre total de transferencias
    synonyms: [índice de fraude, fraud rate]
$$
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.mv_alerts
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Alertas del motor de reglas de fraude y PLD: volumen, falsos positivos, tiempo de atención y recuperación."
source: {catalog}.gold.fact_alert
joins:
  - name: rule
    source: {catalog}.gold.dim_rule
    'on': source.rule_code = rule.rule_code
  - name: customer
    source: {catalog}.gold.dim_customer
    'on': source.customer_id = customer.customer_id
fields:
  - name: mes
    expr: CAST(DATE_TRUNC('MONTH', created_date) AS DATE)
    display_name: Mes
    synonyms: [periodo]
  - name: regla
    expr: rule.rule_name
    display_name: Regla
    synonyms: [regla de alerta, escenario]
  - name: tipo_regla
    expr: rule.rule_type
    display_name: Tipo de regla
    synonyms: [fraude o PLD]
  - name: analista
    expr: analyst
    display_name: Analista
  - name: estatus
    expr: status
    display_name: Estatus de la alerta
  - name: resolucion
    expr: resolution
    display_name: Resolución
    synonyms: [dictamen, resultado]
  - name: segmento
    expr: customer.segment
    display_name: Segmento
measures:
  - name: alertas
    expr: COUNT(1)
    display_name: Alertas
    synonyms: [número de alertas, casos]
  - name: alertas_abiertas
    expr: COUNT(1) FILTER (WHERE status = 'abierta')
    display_name: Alertas abiertas
    synonyms: [pendientes, backlog]
  - name: falsos_positivos
    expr: COUNT(1) FILTER (WHERE is_false_positive)
    display_name: Falsos positivos
  - name: tasa_falsos_positivos
    expr: COUNT(1) FILTER (WHERE is_false_positive) / NULLIF(COUNT(1) FILTER (WHERE status = 'cerrada'), 0)
    display_name: Tasa de falsos positivos
    comment: Falsos positivos entre alertas cerradas
    synonyms: [falsos positivos %, efectividad de reglas]
  - name: fraudes_confirmados
    expr: COUNT(1) FILTER (WHERE is_confirmed_fraud)
    display_name: Fraudes confirmados
  - name: reportes_pld
    expr: COUNT(1) FILTER (WHERE is_pld_report)
    display_name: Reportes PLD
    synonyms: [operaciones inusuales, reportes de operaciones inusuales]
  - name: horas_promedio_atencion
    expr: AVG(hours_to_close)
    display_name: Horas promedio de atención
    synonyms: [tiempo de atención, tiempo de respuesta]
  - name: horas_mediana_atencion
    expr: MEDIAN(hours_to_close)
    display_name: Horas mediana de atención
  - name: monto_alertado
    expr: SUM(transfer_amount)
    display_name: Monto alertado
  - name: monto_recuperado
    expr: SUM(recovered_amount)
    display_name: Monto recuperado
    synonyms: [recuperación]
  - name: tasa_recuperacion
    expr: SUM(recovered_amount) / NULLIF(SUM(transfer_amount) FILTER (WHERE is_confirmed_fraud), 0)
    display_name: Tasa de recuperación
    comment: Monto recuperado entre monto de fraude confirmado
$$
""")

In [0]:
display(spark.sql(f"""
SELECT mes, MEASURE(transferencias) AS transferencias, MEASURE(fraudes_confirmados) AS fraudes,
       MEASURE(tasa_fraude) AS tasa_fraude
FROM {catalog}.gold.mv_fraud
GROUP BY mes
ORDER BY mes
"""))

display(spark.sql(f"""
SELECT regla, tipo_regla, MEASURE(alertas) AS alertas, MEASURE(tasa_falsos_positivos) AS tasa_fp,
       MEASURE(horas_mediana_atencion) AS horas_mediana
FROM {catalog}.gold.mv_alerts
GROUP BY regla, tipo_regla
ORDER BY alertas DESC
"""))